# RBY1 × X-VLA Real Robot Inference

X-VLA FastAPI 서버(`deploy.py`)를 이용한 rby1 실로봇 inference 노트북입니다.

## 연결 구성

| 대상 | 주소 | 비고 |
|------|------|------|
| 실로봇 | `localhost:50051` | 직접 연결 |
| 시뮬레이터 | `localhost:50052` | Docker 포트 매핑 |
| X-VLA 정책 서버 | `XVLA_HOST:XVLA_PORT` | HTTP POST `/act` |

### X-VLA 서버 실행 (별도 터미널)
```bash
python deploy.py \
    --model_path 2toINF/X-VLA-Pt \
    --LoRA_path ./checkpoints/rby1_lora/ckpt-50000 \
    --action_mode auto --real_action_dim 16 \
    --port 8010
```

### 시뮬레이터 실행 (별도 터미널, 선택)
```bash
sudo docker run --rm -it \
  -e DISPLAY=$DISPLAY \
  -v /tmp/.X11-unix:/tmp/.X11-unix \
  -v /exe \
  -p 50052:50051 \
  rainbowroboticsofficial/rby1-sim
```

## 실행 순서
1. Import + Config
2. X-VLA Policy 서버 연결 확인
3. 실로봇 초기 자세 이동 (safe path + head + tool flange 12V)
4. Gripper 초기화 (homing + open)
5. RealSense 카메라 초기화 + 첫 프레임 확인
6. [선택] 시뮬레이터 초기 자세
7. [선택] 시뮬레이터 단일 Action Chunk 테스트 (X-VLA)
8. [선택] 안전 검사 + 실로봇 단일 Action Chunk (X-VLA)
9. 실로봇 Episode Loop (X-VLA)
10. 정리

## OpenPI 대비 차이점
- WebSocket `openpi_client` 대신 **HTTP POST** + `XVLARby1Client` 사용
- action shape: `[T, 20]` → `[:, :16]` trim (client 내부 처리)
- `domain_id=19` (rby1) 전달
- proprio: 16D → 20D zero-pad 후 전송 (client 내부 처리)

## Cell 1 — Import + Config

In [ ]:
from __future__ import annotations

import logging
import sys
import time

from pathlib import Path

import numpy as np
import rby1_sdk as rby

# -- locate X-VLA root --
_root_candidates = [
    Path("/home/hyunjin/rby1_ws/X-VLA"),
    Path.cwd(),
    Path.cwd().parent,
]
XVLA_ROOT = None
for _cand in _root_candidates:
    if (_cand / "evaluation" / "rby1").is_dir():
        XVLA_ROOT = _cand.resolve()
        break
if XVLA_ROOT is None:
    raise RuntimeError("X-VLA root not found. Check workspace path.")
if str(XVLA_ROOT) not in sys.path:
    sys.path.insert(0, str(XVLA_ROOT))

from evaluation.rby1.client import XVLARby1Client

logging.basicConfig(level=logging.INFO, force=True)

# ---------- User Config ----------
XVLA_HOST = "localhost"
XVLA_PORT = 8010

ROBOT_IP = "localhost:50051"
SIM_IP   = "localhost:50052"

PROMPT = "pick up the cup and place it on the plate"

# RealSense serial numbers
CAM_HEAD_SERIAL = "838212070714"
CAM_LEFT_SERIAL = "922612070040"
CAM_RIGHT_SERIAL = "838212074317"

# Test mode — True enables simulator / single-chunk cells
TEST_MODE = False

# Runtime / Control
MAX_HZ = 15.0
NUM_EPISODES = 1
MAX_EPISODE_STEPS = 300

ARM_COMMAND_PRIORITY = 10
ARM_MINIMUM_TIME = 0.1
LOG_ACTION_SEND = True
USE_REMOTE_GRIPPER = True

# Initial pose
ENABLE_INITIAL_POSE = True
SAFE_INIT_PATH = True
INIT_COMMAND_PRIORITY = 10
INIT_DT = 0.05
INIT_HOLD_TIME = 0.2
ENABLE_HEAD_INIT = True
HEAD_INIT_DEG = np.array([0.0, 40.0], dtype=np.float64)
HEAD_INIT = np.deg2rad(HEAD_INIT_DEG)

# Camera resolution for RealSense
IMG_WIDTH  = 640
IMG_HEIGHT = 480
RENDER_SIZE = 224

# X-VLA inference steps (action chunk length)
XVLA_STEPS = 10

# ---------- RTC (Real-Time Chunking) ----------
USE_RTC = True                       # Toggle RTC on/off
RTC_MAX_GUIDANCE_WEIGHT = 10.0        # Max guidance weight (clamped to 1.0 for direct blending)
RTC_EXECUTION_HORIZON   = 10         # Prefix length for guidance weights
RTC_SCHEDULE            = "linear"   # Weight schedule: "linear", "exp", "ones", "zeros"

print("Config loaded")
print(f"ROBOT_IP={ROBOT_IP}  SIM_IP={SIM_IP}")
print(f"XVLA server={XVLA_HOST}:{XVLA_PORT}  PROMPT={PROMPT}")
print(f"USE_RTC={USE_RTC}  RTC_MAX_GUIDANCE_WEIGHT={RTC_MAX_GUIDANCE_WEIGHT}  RTC_EXECUTION_HORIZON={RTC_EXECUTION_HORIZON}  RTC_SCHEDULE={RTC_SCHEDULE}")

## Cell 2 — X-VLA Policy 서버 연결 확인

In [ ]:
xvla_client = XVLARby1Client(
    host=XVLA_HOST,
    port=XVLA_PORT,
    rtc_enabled=USE_RTC,
    rtc_config={
        "max_guidance_weight": RTC_MAX_GUIDANCE_WEIGHT,
        "execution_horizon": RTC_EXECUTION_HORIZON,
        "schedule": RTC_SCHEDULE,
    } if USE_RTC else None,
)

ok = xvla_client.health_check()
print(f"X-VLA server health check: {'OK' if ok else 'FAILED'}")
if not ok:
    raise RuntimeError(
        f"X-VLA server at {XVLA_HOST}:{XVLA_PORT} is not responding.\n"
        "Start it with: python deploy.py --model_path 2toINF/X-VLA-Pt "
        "--LoRA_path ./checkpoints/rby1_lora/ckpt-50000 "
        "--action_mode auto --real_action_dim 16 --port 8010"
    )
print(f"XVLARby1Client ready  url={xvla_client.url}  domain_id={xvla_client.domain_id}  rtc={xvla_client.rtc_enabled}")

## Cell 3 — 실로봇 초기 자세 이동 (safe path + head + tool flange 12V)

In [ ]:
if ENABLE_INITIAL_POSE:
    TORSO_INIT_DEG = np.array([0.0, 20.0, -40.0, 35.0, 0.0, 0.0], dtype=np.float64)
    INIT_RIGHT_DEG = np.array([-24.0, -60.0, 10.0, -120.0, -60.0, 85.0, 0.0], dtype=np.float64)
    INIT_LEFT_DEG = np.array([-24.0, 60.0, -10.0, -120.0, 60.0, 85.0, 0.0], dtype=np.float64)
    TORSO_INIT = np.deg2rad(TORSO_INIT_DEG)
    INIT_RIGHT = np.deg2rad(INIT_RIGHT_DEG)
    INIT_LEFT = np.deg2rad(INIT_LEFT_DEG)

    MID1_RIGHT_DEG = np.array([70.0, -30.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64)
    MID1_LEFT_DEG = np.array([70.0, 30.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64)
    MID2_RIGHT_DEG = np.array([0.0, -15.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64)
    MID2_LEFT_DEG = np.array([0.0, 15.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64)
    MID1_RIGHT = np.deg2rad(MID1_RIGHT_DEG)
    MID1_LEFT = np.deg2rad(MID1_LEFT_DEG)
    MID2_RIGHT = np.deg2rad(MID2_RIGHT_DEG)
    MID2_LEFT = np.deg2rad(MID2_LEFT_DEG)

    TORSO_SLICE = slice(2, 8)
    RIGHT_SLICE = slice(8, 15)
    LEFT_SLICE = slice(15, 22)
    ELBOW_INIT_THRESHOLD_DEG = 90.0

    def _build_body_head_cmd(torso_q, right_q, left_q, head_q, dt):
        body_builder = rby.BodyComponentBasedCommandBuilder()
        if torso_q is not None:
            body_builder = body_builder.set_torso_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(INIT_HOLD_TIME))
                .set_position(np.asarray(torso_q, dtype=np.float64))
                .set_minimum_time(float(dt))
            )
        body_builder = (
            body_builder
            .set_right_arm_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(INIT_HOLD_TIME))
                .set_position(np.asarray(right_q, dtype=np.float64))
                .set_minimum_time(float(dt))
            )
            .set_left_arm_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(INIT_HOLD_TIME))
                .set_position(np.asarray(left_q, dtype=np.float64))
                .set_minimum_time(float(dt))
            )
        )
        cbc = rby.ComponentBasedCommandBuilder().set_body_command(body_builder)
        if head_q is not None:
            cbc = cbc.set_head_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(INIT_HOLD_TIME))
                .set_position(np.asarray(head_q, dtype=np.float64))
                .set_minimum_time(float(dt))
            )
        return rby.RobotCommandBuilder().set_command(cbc)

    def _move_waypoint_stream(robot, stream, name, torso_target, right_target, left_target, head_target, duration):
        q0 = np.asarray(robot.get_state().position, dtype=np.float64)
        start_torso = q0[TORSO_SLICE].copy()
        start_right = q0[RIGHT_SLICE].copy()
        start_left = q0[LEFT_SLICE].copy()
        goal_torso = torso_target if torso_target is not None else start_torso
        steps = max(2, int(np.ceil(duration / INIT_DT)))
        torso_path = np.linspace(start_torso, goal_torso, num=steps)
        right_path = np.linspace(start_right, right_target, num=steps)
        left_path = np.linspace(start_left, left_target, num=steps)
        for i in range(steps):
            cmd = _build_body_head_cmd(torso_path[i], right_path[i], left_path[i], head_target, dt=INIT_DT)
            try:
                stream.send_command(cmd)
            except RuntimeError as exc:
                if "expired" in str(exc).lower():
                    robot.wait_for_control_ready(1000)
                    stream = robot.create_command_stream(priority=INIT_COMMAND_PRIORITY)
                    stream.send_command(cmd)
                else:
                    raise
            time.sleep(INIT_DT)
        q_now = np.asarray(robot.get_state().position, dtype=np.float64)
        print(f"[init-pose] reached {name} | torso_now[1]={q_now[3]:+.4f}, right_now[0]={q_now[8]:+.4f}, left_now[0]={q_now[15]:+.4f}")
        return stream

    robot_init = rby.create_robot(ROBOT_IP, "a") if hasattr(rby, "create_robot") else rby.create_robot_a(ROBOT_IP)
    robot_init.connect()
    assert robot_init.is_connected(), f"Failed to connect robot at {ROBOT_IP}"
    robot_init.power_on(".*")
    robot_init.servo_on(".*")
    robot_init.enable_control_manager()
    try:
        robot_init.cancel_control()
    except Exception:
        pass
    robot_init.wait_for_control_ready(1000)

    q0 = np.asarray(robot_init.get_state().position, dtype=np.float64)
    print("[init-pose] current torso:", np.round(q0[TORSO_SLICE], 3))
    print("[init-pose] current right:", np.round(q0[RIGHT_SLICE], 3))
    print("[init-pose] current left :", np.round(q0[LEFT_SLICE], 3))

    right_elbow_now = float(q0[RIGHT_SLICE][3])
    left_elbow_now  = float(q0[LEFT_SLICE][3])
    elbow_bent = (
        abs(right_elbow_now) > np.deg2rad(ELBOW_INIT_THRESHOLD_DEG)
        or abs(left_elbow_now) > np.deg2rad(ELBOW_INIT_THRESHOLD_DEG)
    )
    print(f"[init-pose] elbow check | right={np.degrees(right_elbow_now):+.1f} deg, left={np.degrees(left_elbow_now):+.1f} deg")

    if SAFE_INIT_PATH:
        if elbow_bent:
            waypoints = [
                ("midpoint2", None,       MID2_RIGHT, MID2_LEFT, 5.0),
                ("initial",   TORSO_INIT, INIT_RIGHT, INIT_LEFT, 3.0),
            ]
        else:
            waypoints = [
                ("midpoint1", None,       MID1_RIGHT, MID1_LEFT, 5.0),
                ("midpoint2", None,       MID2_RIGHT, MID2_LEFT, 5.0),
                ("initial",   TORSO_INIT, INIT_RIGHT, INIT_LEFT, 3.0),
            ]
    else:
        waypoints = [("initial", TORSO_INIT, INIT_RIGHT, INIT_LEFT, 4.0)]

    head_target = HEAD_INIT if ENABLE_HEAD_INIT else None
    stream = robot_init.create_command_stream(priority=INIT_COMMAND_PRIORITY)
    for name, torso_target, right_target, left_target, duration in waypoints:
        print(f"[init-pose] move -> {name} (t={duration:.1f}s)")
        stream = _move_waypoint_stream(
            robot_init, stream, name=name,
            torso_target=torso_target, right_target=right_target,
            left_target=left_target, head_target=head_target, duration=duration,
        )

    time.sleep(0.2)
    q_end = np.asarray(robot_init.get_state().position, dtype=np.float64)
    err_t = float(np.linalg.norm(q_end[TORSO_SLICE] - TORSO_INIT))
    err_r = float(np.linalg.norm(q_end[RIGHT_SLICE] - INIT_RIGHT))
    err_l = float(np.linalg.norm(q_end[LEFT_SLICE] - INIT_LEFT))
    print(f"[init-pose] done | error norm torso={err_t:.6f}, right={err_r:.6f}, left={err_l:.6f}")

    if USE_REMOTE_GRIPPER:
        for _arm in ("right", "left"):
            ok_v = robot_init.set_tool_flange_output_voltage(_arm, 12)
            print(f"[init-pose] tool flange 12V {_arm}: {'OK' if ok_v else 'FAILED'}")
        time.sleep(0.5)

    try:
        robot_init.cancel_control()
    except Exception:
        pass
    if hasattr(robot_init, "disconnect"):
        robot_init.disconnect()

    INITIAL_POSE_DONE = True
else:
    INITIAL_POSE_DONE = False
    print("[init-pose] skipped")

## Cell 4 — Gripper 초기화 (Homing + 열기)

In [ ]:
if not globals().get("INITIAL_POSE_DONE", False):
    raise RuntimeError("Cell 3 (initial pose) must be run first.")
if not globals().get("USE_REMOTE_GRIPPER", False):
    print("[gripper-init] USE_REMOTE_GRIPPER=False -> skipped")
else:
    from evaluation.rby1.remote_gripper import Gripper as RemoteGripper

    # ── 이미 초기화되어 있으면 homing 스킵, 그리퍼만 다시 열기 ──────────────
    _already_init = (
        globals().get("GRIPPER_INIT_DONE", False)
        and "gripper_obj" in globals()
        and gripper_obj is not None
    )
    if _already_init:
        print("[gripper-init] already initialized — skipping homing, reopening grippers")
        try:
            gripper_obj._sock.settimeout(3.0)
            gripper_obj.timeout = 3.0
        except Exception:
            pass
        gripper_obj.set_normalized_target(np.array([1.0, 1.0]), wait_for_reply=False)
        time.sleep(0.3)
        print("[gripper-init] done (fast path)")
    else:
        # ── 전체 초기화 (처음 실행 시) ─────────────────────────────────────
        if "stream" in globals() and stream is not None:
            try:
                del stream
            except Exception:
                pass
            stream = None

        if "gripper_obj" in globals() and gripper_obj is not None:
            try:
                gripper_obj.stop()
            except Exception:
                pass
            gripper_obj = None

        gripper_obj = RemoteGripper()
        print(f"[gripper-init] host={gripper_obj.host}  port={gripper_obj.port}")

        # 1) ping — 짧은 timeout(5s)으로 빠르게 확인
        gripper_obj._sock.settimeout(5.0)
        gripper_obj.timeout = 5.0
        ok_init = gripper_obj.initialize(verbose=True)
        if not ok_init:
            raise RuntimeError("[gripper-init] Gripper server not responding.")
        print("[gripper-init] ping OK")

        # 2) homing — 기계적 동작이므로 timeout 30s 유지
        gripper_obj._sock.settimeout(30.0)
        gripper_obj.timeout = 30.0
        print("[gripper-init] homing... (기계 동작, 최대 30s)")
        ok_homing = gripper_obj.homing()
        if not ok_homing:
            raise RuntimeError("[gripper-init] homing failed")

        _min_q = getattr(gripper_obj, "min_q", None)
        _max_q = getattr(gripper_obj, "max_q", None)
        print(f"[gripper-init] homing done | min_q={_min_q}  max_q={_max_q}")

        # 3) start + 열기 — timeout 3s로 축소
        gripper_obj._sock.settimeout(3.0)
        gripper_obj.timeout = 3.0
        gripper_obj.start()
        gripper_obj.set_normalized_target(np.array([1.0, 1.0]), wait_for_reply=True)
        time.sleep(0.3)

        GRIPPER_INIT_DONE = True
        print("[gripper-init] done")


## Cell 5 — RealSense 카메라 초기화 + 첫 프레임 확인

In [ ]:
import gc
import cv2
import matplotlib.pyplot as plt
import pyrealsense2 as rs

CAM_SERIALS = {
    "head":        CAM_HEAD_SERIAL,
    "left_wrist":  CAM_LEFT_SERIAL,
    "right_wrist": CAM_RIGHT_SERIAL,
}

# (A) 기존 파이프라인 정리
if "_cam_pipelines" in globals() and _cam_pipelines:
    for _p in _cam_pipelines.values():
        try:
            _p.stop()
        except Exception:
            pass
    _cam_pipelines = {}
    print("[camera] 기존 파이프라인 정리 완료")
    time.sleep(1.0)

if "env" in globals() and env is not None:
    try:
        env.close()
    except Exception:
        pass
    env = None
gc.collect()
time.sleep(1.0)

# (B) 카메라 순차 시작 → _cam_pipelines 에 유지 (이후 셀에서 재사용)
print("[camera] 카메라 시작 중...")
_cam_pipelines = {}
for cam_name, serial in CAM_SERIALS.items():
    for attempt in range(1, 4):
        try:
            p = rs.pipeline()
            c = rs.config()
            c.enable_device(serial)
            c.enable_stream(rs.stream.color, IMG_WIDTH, IMG_HEIGHT, rs.format.rgb8, 30)
            p.start(c)
            for _ in range(10):
                p.wait_for_frames(timeout_ms=5000)
            _cam_pipelines[cam_name] = p
            print(f"  [OK] {cam_name:>12} | S/N: {serial} (attempt {attempt})")
            break
        except Exception as _e:
            try:
                p.stop()
            except Exception:
                pass
            if attempt == 3:
                for _pp in _cam_pipelines.values():
                    try:
                        _pp.stop()
                    except Exception:
                        pass
                _cam_pipelines = {}
                raise RuntimeError(
                    f"[camera] '{cam_name}' (S/N: {serial}) 연결 실패: {_e}"
                ) from _e
            print(f"  [RETRY {attempt}/3] {cam_name}: {_e}")
            time.sleep(2.0)
    time.sleep(0.5)

# (C) 테스트 프레임 캡처 (이미 열려 있는 파이프라인에서)
test_frames = {}
for cam_name, p in _cam_pipelines.items():
    frames = p.wait_for_frames(timeout_ms=5000)
    color_frame = frames.get_color_frame()
    if color_frame:
        test_frames[cam_name] = np.asanyarray(color_frame.get_data())

# (D) Visualize
if test_frames:
    fig, axes = plt.subplots(1, len(test_frames), figsize=(5 * len(test_frames), 5))
    if len(test_frames) == 1:
        axes = [axes]
    for ax, (cam_name, img) in zip(axes, test_frames.items()):
        ax.imshow(img)
        ax.set_title(cam_name, fontsize=11)
        ax.axis("off")
    fig.suptitle("RealSense Test Frames", fontsize=13)
    plt.tight_layout()
    plt.show()

CAM_SETUP_DONE = True
print(f"[camera] setup done — {len(_cam_pipelines)} pipelines alive (reusable)")


### Helper: capture one frame from all cameras

Shared utility used by cells 7, 8, 9.

In [ ]:
def _ensure_cameras():
    """Return the global _cam_pipelines dict, validating they are still alive."""
    pipes = globals().get("_cam_pipelines")
    if not pipes:
        raise RuntimeError(
            "[camera] _cam_pipelines is empty. Run the camera init cell (Cell 5) first."
        )
    return pipes


def _capture_images(pipelines=None, resize=RENDER_SIZE):
    """Capture one RGB frame per camera, resize to (resize, resize, 3) uint8."""
    if pipelines is None:
        pipelines = _ensure_cameras()
    images = {}
    for cam_name, p in pipelines.items():
        frames = p.wait_for_frames(timeout_ms=5000)
        color = frames.get_color_frame()
        if color:
            img = np.asanyarray(color.get_data())  # HWC RGB
            if resize is not None:
                img = cv2.resize(img, (resize, resize))
            images[cam_name] = img
    return images


def _get_robot_state_16d(robot, gripper=None):
    """Read robot joint state and return 16-D proprio vector.

    Layout (matches training data assembled in data_conversion.ipynb):
        right_arm(7) [0:7] + left_arm(7) [7:14] + right_gripper(1) [14] + left_gripper(1) [15]
    If *gripper* (RemoteGripper instance) is provided, its normalized state
    is used for index 14 (right) and 15 (left); otherwise they default to 0.0.
    """
    q = np.asarray(robot.get_state().position, dtype=np.float64)
    right_joints = q[8:15]
    left_joints  = q[15:22]

    right_grip_val = 0.0
    left_grip_val  = 0.0
    if gripper is not None:
        try:
            gs = np.asarray(gripper.get_state(), dtype=np.float32).reshape(-1)
            if gs.size >= 2:
                right_grip_val = float(gs[0])
                left_grip_val  = float(gs[1])
        except Exception:
            pass

    return np.concatenate([
        right_joints, left_joints, [right_grip_val], [left_grip_val]
    ]).astype(np.float32)


print("Helpers defined")

## Cell 6 — [선택] 시뮬레이터 초기 자세 이동

In [ ]:
if not globals().get("TEST_MODE", False):
    print("[sim-init] TEST_MODE=False -> skipped")
else:
    SIM_ADDR = SIM_IP
    SIM_INIT_DT = 0.05
    SIM_INIT_HOLD_TIME = 0.5

    TORSO_INIT_SIM = np.deg2rad(np.array([0.0, 20.0, -40.0, 35.0, 0.0, 0.0], dtype=np.float64))
    INIT_RIGHT_SIM = np.deg2rad(np.array([-24.0, -60.0, 10.0, -120.0, -60.0, 85.0, 0.0], dtype=np.float64))
    INIT_LEFT_SIM  = np.deg2rad(np.array([-24.0,  60.0, -10.0, -120.0,  60.0, 85.0, 0.0], dtype=np.float64))
    MID1_RIGHT_SIM = np.deg2rad(np.array([70.0, -30.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64))
    MID1_LEFT_SIM  = np.deg2rad(np.array([70.0,  30.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64))
    MID2_RIGHT_SIM = np.deg2rad(np.array([0.0, -15.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64))
    MID2_LEFT_SIM  = np.deg2rad(np.array([0.0,  15.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64))

    TORSO_SLICE_SIM = slice(2, 8)
    RIGHT_SLICE_SIM = slice(8, 15)
    LEFT_SLICE_SIM  = slice(15, 22)

    def _build_body_cmd_sim(torso_q, right_q, left_q, dt):
        body_builder = rby.BodyComponentBasedCommandBuilder()
        if torso_q is not None:
            body_builder = body_builder.set_torso_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(SIM_INIT_HOLD_TIME))
                .set_position(np.asarray(torso_q, dtype=np.float64))
                .set_minimum_time(float(dt))
            )
        body_builder = (
            body_builder
            .set_right_arm_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(SIM_INIT_HOLD_TIME))
                .set_position(np.asarray(right_q, dtype=np.float64))
                .set_minimum_time(float(dt))
            )
            .set_left_arm_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(SIM_INIT_HOLD_TIME))
                .set_position(np.asarray(left_q, dtype=np.float64))
                .set_minimum_time(float(dt))
            )
        )
        return rby.RobotCommandBuilder().set_command(
            rby.ComponentBasedCommandBuilder().set_body_command(body_builder)
        )

    def _move_waypoint_sim(robot, stream, torso_target, right_target, left_target, duration):
        q0 = np.asarray(robot.get_state().position, dtype=np.float64)
        start_torso = q0[TORSO_SLICE_SIM].copy()
        start_right = q0[RIGHT_SLICE_SIM].copy()
        start_left  = q0[LEFT_SLICE_SIM].copy()
        goal_torso = torso_target if torso_target is not None else start_torso
        steps = max(2, int(np.ceil(duration / SIM_INIT_DT)))
        torso_path = np.linspace(start_torso, goal_torso, num=steps)
        right_path = np.linspace(start_right, right_target, num=steps)
        left_path  = np.linspace(start_left,  left_target,  num=steps)
        for i in range(steps):
            cmd_wp = _build_body_cmd_sim(torso_path[i], right_path[i], left_path[i], dt=SIM_INIT_DT)
            try:
                stream.send_command(cmd_wp)
            except RuntimeError as exc:
                if "expired" in str(exc).lower():
                    robot.wait_for_control_ready(1000)
                    stream = robot.create_command_stream(priority=10)
                    stream.send_command(cmd_wp)
                else:
                    raise
            time.sleep(SIM_INIT_DT)
        return stream

    robot_sim = rby.create_robot_a(SIM_ADDR) if hasattr(rby, "create_robot_a") else rby.create_robot(SIM_ADDR, "a")
    robot_sim.connect()
    assert robot_sim.is_connected(), f"Failed to connect simulator at {SIM_ADDR}"
    robot_sim.power_on(".*")
    robot_sim.servo_on(".*")
    robot_sim.reset_fault_control_manager()
    if not robot_sim.enable_control_manager():
        raise RuntimeError("[sim-init] Failed to enable control manager")

    stream_sim = robot_sim.create_command_stream(priority=10)
    sim_waypoints = [
        (None,           MID1_RIGHT_SIM, MID1_LEFT_SIM, 5.0),
        (None,           MID2_RIGHT_SIM, MID2_LEFT_SIM, 5.0),
        (TORSO_INIT_SIM, INIT_RIGHT_SIM, INIT_LEFT_SIM, 3.0),
    ]
    for torso_t, right_t, left_t, dur_t in sim_waypoints:
        stream_sim = _move_waypoint_sim(robot_sim, stream_sim, torso_t, right_t, left_t, dur_t)

    time.sleep(0.2)
    q_init_sim = np.asarray(robot_sim.get_state().position, dtype=np.float64)
    print("[sim-init] right:", np.round(q_init_sim[8:15], 4))
    print("[sim-init] left :", np.round(q_init_sim[15:22], 4))

    try:
        robot_sim.cancel_control()
    except Exception:
        pass
    if hasattr(robot_sim, "disconnect"):
        robot_sim.disconnect()

    SIM_INITIAL_POSE_DONE = True
    print("[sim-init] done")

## Cell 7 — [선택] 시뮬레이터 단일 Action Chunk 테스트 (X-VLA)

In [ ]:
if not globals().get("TEST_MODE", False):
    print("[sim-chunk] TEST_MODE=False -> skipped")
else:
    if not globals().get("CAM_SETUP_DONE", False):
        raise RuntimeError("Run Cell 5 (camera setup) first.")
    if not globals().get("SIM_INITIAL_POSE_DONE", False):
        raise RuntimeError("Run Cell 6 (sim initial pose) first.")

    SIM_ADDR            = SIM_IP
    SIM_CHUNK_DT        = 0.02
    SIM_CHUNK_HOLD_TIME = 0.2
    SIM_CHUNK_PRIORITY  = 10

    # 1) Capture images from real cameras + sim robot state -> inference
    cam_pipes = _start_cameras()
    imgs = _capture_images(cam_pipes)

    robot_sim_chunk = rby.create_robot_a(SIM_ADDR) if hasattr(rby, "create_robot_a") else rby.create_robot(SIM_ADDR, "a")
    robot_sim_chunk.connect()
    assert robot_sim_chunk.is_connected()

    _grip = globals().get("gripper_obj")
    proprio = _get_robot_state_16d(robot_sim_chunk, gripper=_grip)

    actions = xvla_client.infer(
        head_img=imgs["head"],
        left_img=imgs["left_wrist"],
        right_img=imgs["right_wrist"],
        proprio_16d=proprio,
        instruction=PROMPT,
        steps=XVLA_STEPS,
    )
    print(f"[sim-chunk] action shape: {actions.shape}")

    # Debug: verify delta→absolute conversion
    print(f"[sim-chunk] proprio (first 7, right arm):  {np.round(proprio[0:7], 4)}")
    print(f"[sim-chunk] action[0] (first 7, right arm): {np.round(actions[0, 0:7], 4)}")
    print(f"[sim-chunk] delta = action - proprio (right): {np.round(actions[0, 0:7] - proprio[0:7], 4)}")

    right_targets = actions[:, 0:7]
    left_targets  = actions[:, 7:14]

    # 2) Send to simulator
    robot_sim_chunk.power_on(".*")
    robot_sim_chunk.servo_on(".*")
    robot_sim_chunk.reset_fault_control_manager()
    if not robot_sim_chunk.enable_control_manager():
        raise RuntimeError("[sim-chunk] Failed to enable control manager")

    q0 = np.asarray(robot_sim_chunk.get_state().position, dtype=np.float64)
    torso_hold = q0[2:8].copy()
    sim_right_start = q0[8:15].copy()
    sim_left_start  = q0[15:22].copy()

    stream_sim_chunk = robot_sim_chunk.create_command_stream(priority=SIM_CHUNK_PRIORITY)

    for i in range(actions.shape[0]):
        cmd = rby.RobotCommandBuilder().set_command(
            rby.ComponentBasedCommandBuilder().set_body_command(
                rby.BodyComponentBasedCommandBuilder()
                .set_torso_command(
                    rby.JointPositionCommandBuilder()
                    .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(SIM_CHUNK_HOLD_TIME))
                    .set_position(torso_hold.astype(np.float64))
                    .set_minimum_time(SIM_CHUNK_DT)
                )
                .set_right_arm_command(
                    rby.JointPositionCommandBuilder()
                    .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(SIM_CHUNK_HOLD_TIME))
                    .set_position(right_targets[i].astype(np.float64))
                    .set_minimum_time(SIM_CHUNK_DT)
                )
                .set_left_arm_command(
                    rby.JointPositionCommandBuilder()
                    .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(SIM_CHUNK_HOLD_TIME))
                    .set_position(left_targets[i].astype(np.float64))
                    .set_minimum_time(SIM_CHUNK_DT)
                )
            )
        )
        try:
            stream_sim_chunk.send_command(cmd)
        except RuntimeError as exc:
            if "expired" in str(exc).lower():
                robot_sim_chunk.wait_for_control_ready(1000)
                stream_sim_chunk = robot_sim_chunk.create_command_stream(priority=SIM_CHUNK_PRIORITY)
                stream_sim_chunk.send_command(cmd)
            else:
                raise

        if i == 0 or (i + 1) % 5 == 0 or (i + 1) == actions.shape[0]:
            q_now = np.asarray(robot_sim_chunk.get_state().position, dtype=np.float64)
            err_r = float(np.linalg.norm(right_targets[i] - q_now[8:15]))
            err_l = float(np.linalg.norm(left_targets[i]  - q_now[15:22]))
            print(f"  step {i+1}/{actions.shape[0]} | err(r={err_r:.4f}, l={err_l:.4f})")
        time.sleep(SIM_CHUNK_DT)

    time.sleep(0.5)
    q_after = np.asarray(robot_sim_chunk.get_state().position, dtype=np.float64)
    print(f"[sim-chunk] movement norm | right={np.linalg.norm(q_after[8:15] - sim_right_start):.4f}, left={np.linalg.norm(q_after[15:22] - sim_left_start):.4f}")

    _stop_cameras(cam_pipes)
    try:
        robot_sim_chunk.cancel_control()
    except Exception:
        pass
    if hasattr(robot_sim_chunk, "disconnect"):
        robot_sim_chunk.disconnect()

## Cell 8 — [선택] 안전 검사 + 실로봇 단일 Action Chunk (X-VLA)

In [ ]:
if not globals().get("TEST_MODE", False):
    print("[safe-chunk] TEST_MODE=False -> skipped")
else:
    if not globals().get("CAM_SETUP_DONE", False):
        raise RuntimeError("Run Cell 5 first.")
    if not globals().get("INITIAL_POSE_DONE", False):
        raise RuntimeError("Run Cell 3 first.")

    SAFETY_MAX_TOTAL_NORM = 1.0
    SAFETY_MAX_STEP_DELTA = 0.15
    SAFETY_MAX_JOINT_DEG  = 30.0
    SAFETY_OVERRIDE       = True
    SAFE_DT        = 0.02
    SAFE_HOLD_TIME = 0.5
    SAFE_PRIORITY  = ARM_COMMAND_PRIORITY

    # 1) Inference
    cam_pipes = _ensure_cameras()
    imgs = _capture_images(cam_pipes)

    robot_safe = rby.create_robot(ROBOT_IP, "a") if hasattr(rby, "create_robot") else rby.create_robot_a(ROBOT_IP)
    robot_safe.connect()
    assert robot_safe.is_connected()

    _grip = globals().get("gripper_obj")
    proprio = _get_robot_state_16d(robot_safe, gripper=_grip)

    actions = xvla_client.infer(
        head_img=imgs["head"],
        left_img=imgs["left_wrist"],
        right_img=imgs["right_wrist"],
        proprio_16d=proprio,
        instruction=PROMPT,
        steps=XVLA_STEPS,
    )
    print(f"[safe-chunk] action shape: {actions.shape}")

    right_targets = actions[:, 0:7]
    left_targets  = actions[:, 7:14]
    right_gripper = actions[:, 14]
    left_gripper  = actions[:, 15]
    T = actions.shape[0]

    q_safe    = np.asarray(robot_safe.get_state().position, dtype=np.float64)
    torso_hold = q_safe[2:8].copy()
    cur_right  = q_safe[8:15].copy()
    cur_left   = q_safe[15:22].copy()

    # 2) Safety check
    violations = []
    total_r = float(np.linalg.norm(right_targets[-1] - right_targets[0]))
    total_l = float(np.linalg.norm(left_targets[-1]  - left_targets[0]))
    if total_r > SAFETY_MAX_TOTAL_NORM:
        violations.append(f"right total norm {total_r:.4f} > {SAFETY_MAX_TOTAL_NORM}")
    if total_l > SAFETY_MAX_TOTAL_NORM:
        violations.append(f"left total norm {total_l:.4f} > {SAFETY_MAX_TOTAL_NORM}")

    r_deltas = np.linalg.norm(np.diff(right_targets, axis=0), axis=1)
    l_deltas = np.linalg.norm(np.diff(left_targets,  axis=0), axis=1)
    if len(r_deltas) and r_deltas.max() > SAFETY_MAX_STEP_DELTA:
        violations.append(f"right max step delta {r_deltas.max():.4f} > {SAFETY_MAX_STEP_DELTA}")
    if len(l_deltas) and l_deltas.max() > SAFETY_MAX_STEP_DELTA:
        violations.append(f"left max step delta {l_deltas.max():.4f} > {SAFETY_MAX_STEP_DELTA}")

    disp_r = float(np.degrees(np.abs(right_targets - cur_right[None, :]).max()))
    disp_l = float(np.degrees(np.abs(left_targets  - cur_left[None,  :]).max()))
    if disp_r > SAFETY_MAX_JOINT_DEG:
        violations.append(f"right max joint displacement {disp_r:.2f} deg > {SAFETY_MAX_JOINT_DEG}")
    if disp_l > SAFETY_MAX_JOINT_DEG:
        violations.append(f"left max joint displacement {disp_l:.2f} deg > {SAFETY_MAX_JOINT_DEG}")

    if violations:
        print("Safety violations:")
        for v in violations:
            print(f"  - {v}")
        if not SAFETY_OVERRIDE:
            robot_safe.disconnect()
            raise RuntimeError("Safety check failed. Set SAFETY_OVERRIDE=True to override.")
        print("SAFETY_OVERRIDE=True -> proceeding")
    else:
        print("[safe-chunk] Safety check passed")

    # 3) Send to real robot
    robot_safe.power_on(".*")
    robot_safe.servo_on(".*")
    robot_safe.reset_fault_control_manager()
    if not robot_safe.enable_control_manager():
        robot_safe.disconnect()
        raise RuntimeError("[safe-chunk] Failed to enable control manager")

    stream_safe = robot_safe.create_command_stream(priority=SAFE_PRIORITY)

    for i in range(T):
        cmd = rby.RobotCommandBuilder().set_command(
            rby.ComponentBasedCommandBuilder().set_body_command(
                rby.BodyComponentBasedCommandBuilder()
                .set_torso_command(
                    rby.JointPositionCommandBuilder()
                    .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(SAFE_HOLD_TIME))
                    .set_position(torso_hold.astype(np.float64))
                    .set_minimum_time(SAFE_DT)
                )
                .set_right_arm_command(
                    rby.JointPositionCommandBuilder()
                    .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(SAFE_HOLD_TIME))
                    .set_position(right_targets[i].astype(np.float64))
                    .set_minimum_time(SAFE_DT)
                )
                .set_left_arm_command(
                    rby.JointPositionCommandBuilder()
                    .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(SAFE_HOLD_TIME))
                    .set_position(left_targets[i].astype(np.float64))
                    .set_minimum_time(SAFE_DT)
                )
            )
        )
        try:
            stream_safe.send_command(cmd)
        except RuntimeError as exc:
            if "expired" in str(exc).lower():
                robot_safe.wait_for_control_ready(1000)
                stream_safe = robot_safe.create_command_stream(priority=SAFE_PRIORITY)
                stream_safe.send_command(cmd)
            else:
                raise

        if USE_REMOTE_GRIPPER and globals().get("gripper_obj") is not None:
            try:
                gripper_obj.set_normalized_target(
                    np.array([float(right_gripper[i]), float(left_gripper[i])]),
                    wait_for_reply=False,
                )
            except Exception:
                pass

        if i == 0 or (i + 1) % 5 == 0 or (i + 1) == T:
            q_now = np.asarray(robot_safe.get_state().position, dtype=np.float64)
            err_r = float(np.linalg.norm(right_targets[i] - q_now[8:15]))
            err_l = float(np.linalg.norm(left_targets[i]  - q_now[15:22]))
            print(f"  step {i+1}/{T} | err(r={err_r:.4f}, l={err_l:.4f}) | grip(R={right_gripper[i]:+.3f}, L={left_gripper[i]:+.3f})")
        time.sleep(SAFE_DT)

    time.sleep(0.5)
    q_final = np.asarray(robot_safe.get_state().position, dtype=np.float64)
    print(f"[safe-chunk] movement norm | right={np.linalg.norm(q_final[8:15] - cur_right):.4f}, left={np.linalg.norm(q_final[15:22] - cur_left):.4f}")

    try:
        robot_safe.cancel_control()
    except Exception:
        pass
    if hasattr(robot_safe, "disconnect"):
        robot_safe.disconnect()

## Cell 9 — 실로봇 Episode Loop (X-VLA)

In [ ]:
if not globals().get("CAM_SETUP_DONE", False):
    raise RuntimeError("Run Cell 5 first.")
if not globals().get("INITIAL_POSE_DONE", False):
    raise RuntimeError("Run Cell 3 first.")

REPLAY_DT          = 0.05 # 0.1
REPLAY_PRIORITY    = ARM_COMMAND_PRIORITY
REPLAY_HOLD_TIME   = 5.0   # must be >> inference time so the arm doesn't fall back between chunks
EPISODE_LENGTH     = 800
EXECUTE_CHUNK_SIZE = 20  # Execute this many steps per inference, then re-infer

CONTROL_MODE = "position"  # "position" or "impedance"

# Impedance params (only used when CONTROL_MODE == "impedance")
ARM_STIFFNESS       = np.array([80.0, 80.0, 80.0, 80.0, 80.0, 80.0, 40.0])
ARM_DAMPING_RATIO   = 1.0
ARM_TORQUE_LIMIT    = np.array([30.0] * 7)
TORSO_STIFFNESS     = np.array([400.0] * 6)
TORSO_DAMPING_RATIO = 1.0
TORSO_TORQUE_LIMIT  = np.array([500.0] * 6)

assert CONTROL_MODE in ("position", "impedance")

def _build_arm_cmd(position, is_impedance):
    if is_impedance:
        return (
            rby.JointImpedanceControlCommandBuilder()
            .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(REPLAY_HOLD_TIME))
            .set_position(position.astype(np.float64))
            .set_minimum_time(REPLAY_DT)
            .set_stiffness(ARM_STIFFNESS)
            .set_damping_ratio(ARM_DAMPING_RATIO)
            .set_torque_limit(ARM_TORQUE_LIMIT)
        )
    return (
        rby.JointPositionCommandBuilder()
        .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(REPLAY_HOLD_TIME))
        .set_position(position.astype(np.float64))
        .set_minimum_time(REPLAY_DT)
    )

def _build_torso_cmd(position, is_impedance):
    if is_impedance:
        return (
            rby.JointImpedanceControlCommandBuilder()
            .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(REPLAY_HOLD_TIME))
            .set_position(position.astype(np.float64))
            .set_minimum_time(REPLAY_DT)
            .set_stiffness(TORSO_STIFFNESS)
            .set_damping_ratio(TORSO_DAMPING_RATIO)
            .set_torque_limit(TORSO_TORQUE_LIMIT)
        )
    return (
        rby.JointPositionCommandBuilder()
        .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(REPLAY_HOLD_TIME))
        .set_position(position.astype(np.float64))
        .set_minimum_time(REPLAY_DT)
    )

_use_impedance = (CONTROL_MODE == "impedance")

# Connect robot
robot = rby.create_robot(ROBOT_IP, "a") if hasattr(rby, "create_robot") else rby.create_robot_a(ROBOT_IP)
robot.connect()
assert robot.is_connected()
robot.power_on(".*")
robot.servo_on(".*")
robot.reset_fault_control_manager()
if not robot.enable_control_manager():
    robot.disconnect()
    raise RuntimeError("[episode] Failed to enable control manager")

q_init     = np.asarray(robot.get_state().position, dtype=np.float64)
ep_start_r = q_init[8:15].copy()
ep_start_l = q_init[15:22].copy()
print(f"[episode] EPISODE_LENGTH={EPISODE_LENGTH}, EXECUTE_CHUNK_SIZE={EXECUTE_CHUNK_SIZE}, CONTROL_MODE={CONTROL_MODE}")
print(f"[episode] REPLAY_HOLD_TIME={REPLAY_HOLD_TIME}s  (must exceed inference latency)")
print(f"[episode] episode start | right={np.round(ep_start_r, 4)}, left={np.round(ep_start_l, 4)}")

stream      = robot.create_command_stream(priority=REPLAY_PRIORITY)
total_steps = 0
chunk_idx   = 0
send_errors = 0

# --------------------------------------------------
# Logging buffers
# --------------------------------------------------
_log_cmd_right    = []
_log_cmd_left     = []
_log_cmd_grip_r   = []
_log_cmd_grip_l   = []
_log_actual_right = []
_log_actual_left  = []
_log_actual_step  = []
_log_chunk_start  = []
_log_obs_images   = []
_log_full_chunks  = []  # full predicted chunk per inference (including un-executed tail)

# Reset server-side RTC state for fresh episode
if xvla_client.rtc_enabled:
    xvla_client.reset_rtc()
    print(f"[episode] RTC enabled (gw={RTC_MAX_GUIDANCE_WEIGHT}, horizon={RTC_EXECUTION_HORIZON}, sched={RTC_SCHEDULE})")
else:
    print("[episode] RTC disabled")

# Use persistent cameras from Cell 5
cam_pipes = _ensure_cameras()

try:
    while total_steps < EPISODE_LENGTH:
        remaining = EPISODE_LENGTH - total_steps

        # 1) Capture observation + inference
        imgs = _capture_images(cam_pipes)
        _grip = globals().get("gripper_obj")
        proprio = _get_robot_state_16d(robot, gripper=_grip)

        # Observation image snapshot
        _img_snap = {"step": total_steps}
        for _k in ("head", "left_wrist", "right_wrist"):
            if _k in imgs:
                _img_snap[_k] = imgs[_k].copy()
        _log_obs_images.append(_img_snap)

        actions = xvla_client.infer(
            head_img=imgs["head"],
            left_img=imgs["left_wrist"],
            right_img=imgs["right_wrist"],
            proprio_16d=proprio,
            instruction=PROMPT,
            steps=XVLA_STEPS,
            execute_chunk_size=EXECUTE_CHUNK_SIZE if xvla_client.rtc_enabled else None,
        )

        right_targets     = actions[:, 0:7]
        left_targets      = actions[:, 7:14]
        right_gripper_out = actions[:, 14]
        left_gripper_out  = actions[:, 15]

        # Save the FULL predicted chunk (including un-executed tail) for continuity analysis
        _log_full_chunks.append({
            "step":  total_steps,
            "right": actions[:, 0:7].copy(),
            "left":  actions[:, 7:14].copy(),
        })

        q_now      = np.asarray(robot.get_state().position, dtype=np.float64)
        torso_hold = q_now[2:8].copy()

        chunk_size    = actions.shape[0]
        execute_size  = EXECUTE_CHUNK_SIZE if EXECUTE_CHUNK_SIZE is not None else chunk_size
        steps_to_send = min(execute_size, remaining, chunk_size)
        total_cmd_norm = float(np.linalg.norm(right_targets[steps_to_send - 1] - right_targets[0]))

        _log_chunk_start.append(total_steps)
        print(
            f"[episode] chunk {chunk_idx+1} | "
            f"steps {total_steps+1}~{total_steps+steps_to_send}/{EPISODE_LENGTH} "
            f"(execute {steps_to_send}/{chunk_size}) | cmd_norm={total_cmd_norm:.4f}"
        )

        # 2) Send chunk
        for i in range(steps_to_send):
            cmd = rby.RobotCommandBuilder().set_command(
                rby.ComponentBasedCommandBuilder().set_body_command(
                    rby.BodyComponentBasedCommandBuilder()
                    .set_torso_command(_build_torso_cmd(torso_hold, _use_impedance))
                    .set_right_arm_command(_build_arm_cmd(right_targets[i], _use_impedance))
                    .set_left_arm_command(_build_arm_cmd(left_targets[i], _use_impedance))
                )
            )
            try:
                stream.send_command(cmd)
            except RuntimeError as exc:
                send_errors += 1
                if "expired" in str(exc).lower():
                    robot.wait_for_control_ready(1000)
                    stream = robot.create_command_stream(priority=REPLAY_PRIORITY)
                    stream.send_command(cmd)
                else:
                    raise

            if USE_REMOTE_GRIPPER and globals().get("gripper_obj") is not None:
                try:
                    gripper_obj.set_normalized_target(
                        np.array([float(right_gripper_out[i]), float(left_gripper_out[i])]),
                        wait_for_reply=False,
                    )
                except Exception:
                    pass

            # Command logging
            _log_cmd_right.append(right_targets[i].copy())
            _log_cmd_left.append(left_targets[i].copy())
            _log_cmd_grip_r.append(float(right_gripper_out[i]))
            _log_cmd_grip_l.append(float(left_gripper_out[i]))

            if i == 0 or (i + 1) % 10 == 0 or (i + 1) == steps_to_send:
                q_diag = np.asarray(robot.get_state().position, dtype=np.float64)
                err_r = float(np.linalg.norm(right_targets[i] - q_diag[8:15]))
                err_l = float(np.linalg.norm(left_targets[i]  - q_diag[15:22]))
                moved_r = float(np.linalg.norm(q_diag[8:15]  - ep_start_r))
                moved_l = float(np.linalg.norm(q_diag[15:22] - ep_start_l))
                _log_actual_right.append(q_diag[8:15].copy())
                _log_actual_left.append(q_diag[15:22].copy())
                _log_actual_step.append(total_steps + i)
                mark = " [IMP]" if _use_impedance else ""
                print(
                    f"  step {total_steps+i+1:4d}/{EPISODE_LENGTH}{mark} "
                    f"| tracking_err(r={err_r:.4f}, l={err_l:.4f}) "
                    f"| moved(r={moved_r:.4f}, l={moved_l:.4f}) "
                    f"| grip(R={right_gripper_out[i]:+.3f}, L={left_gripper_out[i]:+.3f})"
                )

            time.sleep(REPLAY_DT)

        total_steps += steps_to_send
        chunk_idx   += 1

finally:
    pass  # cameras stay alive for next episode

# Final report
time.sleep(0.5)
q_after    = np.asarray(robot.get_state().position, dtype=np.float64)
obs_norm_r = float(np.linalg.norm(q_after[8:15]  - ep_start_r))
obs_norm_l = float(np.linalg.norm(q_after[15:22] - ep_start_l))

print(f"\n[episode] === episode done ({CONTROL_MODE} mode) ===")
print(f"[episode] total_steps={total_steps}, chunks={chunk_idx}, send_errors={send_errors}")
print(f"[episode] ep_start right : {np.round(ep_start_r, 4)}")
print(f"[episode] q_after  right : {np.round(q_after[8:15],  4)}")
print(f"[episode] ep_start left  : {np.round(ep_start_l, 4)}")
print(f"[episode] q_after  left  : {np.round(q_after[15:22], 4)}")
print(f"[episode] total movement norm | right={obs_norm_r:.4f}, left={obs_norm_l:.4f}")
print(f"[episode] log: {len(_log_cmd_right)} steps, {len(_log_chunk_start)} chunks, {len(_log_obs_images)} obs, {len(_log_full_chunks)} full chunks")

try:
    robot.cancel_control()
except Exception:
    pass
if hasattr(robot, "disconnect"):
    robot.disconnect()

# [Initial Pose] (episode 후 복귀용)

In [ ]:
# ---------- Step 2: initial pose 이동 (safe path + head) ----------
if ENABLE_INITIAL_POSE:
    # data-collection 초기 자세
    TORSO_INIT_DEG = np.array([0.0, 20.0, -40.0, 35.0, 0.0, 0.0], dtype=np.float64)
    INIT_RIGHT_DEG = np.array([-24.0, -60.0, 10.0, -120.0, -60.0, 85.0, 0.0], dtype=np.float64)
    INIT_LEFT_DEG = np.array([-24.0, 60.0, -10.0, -120.0, 60.0, 85.0, 0.0], dtype=np.float64)
    TORSO_INIT = np.deg2rad(TORSO_INIT_DEG)
    INIT_RIGHT = np.deg2rad(INIT_RIGHT_DEG)
    INIT_LEFT = np.deg2rad(INIT_LEFT_DEG)

    MID1_RIGHT_DEG = np.array([70.0, -30.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64)
    MID1_LEFT_DEG = np.array([70.0, 30.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64)
    MID2_RIGHT_DEG = np.array([0.0, -15.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64)
    MID2_LEFT_DEG = np.array([0.0, 15.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64)
    MID1_RIGHT = np.deg2rad(MID1_RIGHT_DEG)
    MID1_LEFT = np.deg2rad(MID1_LEFT_DEG)
    MID2_RIGHT = np.deg2rad(MID2_RIGHT_DEG)
    MID2_LEFT = np.deg2rad(MID2_LEFT_DEG)

    TORSO_SLICE = slice(2, 8)
    RIGHT_SLICE = slice(8, 15)
    LEFT_SLICE = slice(15, 22)
    ELBOW_INIT_THRESHOLD_DEG = 90.0

    def _build_body_head_cmd(torso_q, right_q, left_q, head_q, dt):
        body_builder = rby.BodyComponentBasedCommandBuilder()
        if torso_q is not None:
            body_builder = body_builder.set_torso_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(INIT_HOLD_TIME))
                .set_position(np.asarray(torso_q, dtype=np.float64))
                .set_minimum_time(float(dt))
            )
        body_builder = (
            body_builder
            .set_right_arm_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(INIT_HOLD_TIME))
                .set_position(np.asarray(right_q, dtype=np.float64))
                .set_minimum_time(float(dt))
            )
            .set_left_arm_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(INIT_HOLD_TIME))
                .set_position(np.asarray(left_q, dtype=np.float64))
                .set_minimum_time(float(dt))
            )
        )
        cbc = rby.ComponentBasedCommandBuilder().set_body_command(body_builder)
        if head_q is not None:
            cbc = cbc.set_head_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(INIT_HOLD_TIME))
                .set_position(np.asarray(head_q, dtype=np.float64))
                .set_minimum_time(float(dt))
            )
        return rby.RobotCommandBuilder().set_command(cbc)

    def _move_waypoint_stream(robot, stream, name, torso_target, right_target, left_target, head_target, duration):
        q0 = np.asarray(robot.get_state().position, dtype=np.float64)
        start_torso = q0[TORSO_SLICE].copy()
        start_right = q0[RIGHT_SLICE].copy()
        start_left = q0[LEFT_SLICE].copy()
        goal_torso = torso_target if torso_target is not None else start_torso
        steps = max(2, int(np.ceil(duration / INIT_DT)))
        torso_path = np.linspace(start_torso, goal_torso, num=steps)
        right_path = np.linspace(start_right, right_target, num=steps)
        left_path = np.linspace(start_left, left_target, num=steps)
        for i in range(steps):
            cmd = _build_body_head_cmd(torso_path[i], right_path[i], left_path[i], head_target, dt=INIT_DT)
            try:
                stream.send_command(cmd)
            except RuntimeError as exc:
                if "expired" in str(exc).lower():
                    robot.wait_for_control_ready(1000)
                    stream = robot.create_command_stream(priority=INIT_COMMAND_PRIORITY)
                    stream.send_command(cmd)
                else:
                    raise
            time.sleep(INIT_DT)
        q_now = np.asarray(robot.get_state().position, dtype=np.float64)
        print(f"[init-pose] reached {name} | torso_now[1]={q_now[3]:+.4f}, right_now[0]={q_now[8]:+.4f}, left_now[0]={q_now[15]:+.4f}")
        return stream

    robot_init = rby.create_robot(ROBOT_IP, "a") if hasattr(rby, "create_robot") else rby.create_robot_a(ROBOT_IP)
    robot_init.connect()
    assert robot_init.is_connected(), f"Failed to connect robot at {ROBOT_IP}"

    robot_init.power_on(".*")
    robot_init.servo_on(".*")
    robot_init.enable_control_manager()

    try:
        robot_init.cancel_control()
    except Exception:
        pass
    robot_init.wait_for_control_ready(1000)

    q0 = np.asarray(robot_init.get_state().position, dtype=np.float64)
    print("[init-pose] current torso:", np.round(q0[TORSO_SLICE], 3))
    print("[init-pose] current right:", np.round(q0[RIGHT_SLICE], 3))
    print("[init-pose] current left :", np.round(q0[LEFT_SLICE], 3))
    if ENABLE_HEAD_INIT:
        print("[init-pose] target head(deg):", np.round(HEAD_INIT_DEG, 2))

    right_elbow_now = float(q0[RIGHT_SLICE][3])
    left_elbow_now  = float(q0[LEFT_SLICE][3])
    elbow_bent = (
        abs(right_elbow_now) > np.deg2rad(ELBOW_INIT_THRESHOLD_DEG)
        or abs(left_elbow_now) > np.deg2rad(ELBOW_INIT_THRESHOLD_DEG)
    )
    print(
        f"[init-pose] elbow check | right={np.degrees(right_elbow_now):+.1f}, "
        f"left={np.degrees(left_elbow_now):+.1f}  "
        f"-> {'near-INITIAL (bent)' if elbow_bent else 'near-ZERO (straight)'}"
    )

    if SAFE_INIT_PATH:
        if elbow_bent:
            waypoints = [
                ("midpoint2", None,       MID2_RIGHT, MID2_LEFT, 5.0),
                ("initial",   TORSO_INIT, INIT_RIGHT, INIT_LEFT, 3.0),
            ]
            print("[init-pose] near-initial detected: midpoint2 -> initial")
        else:
            waypoints = [
                ("midpoint1", None,       MID1_RIGHT, MID1_LEFT, 7.0),
                ("midpoint2", None,       MID2_RIGHT, MID2_LEFT, 7.0),
                ("initial",   TORSO_INIT, INIT_RIGHT, INIT_LEFT, 2.0),
            ]
            print("[init-pose] near-zero detected: midpoint1 -> midpoint2 -> initial")
    else:
        waypoints = [("initial", TORSO_INIT, INIT_RIGHT, INIT_LEFT, 4.0)]
        print("[init-pose] SAFE_INIT_PATH disabled: direct -> initial")

    head_target = HEAD_INIT if ENABLE_HEAD_INIT else None
    stream = robot_init.create_command_stream(priority=INIT_COMMAND_PRIORITY)
    for name, torso_target, right_target, left_target, duration in waypoints:
        print(f"[init-pose] move -> {name} (t={duration:.1f}s)")
        stream = _move_waypoint_stream(
            robot_init, stream, name=name,
            torso_target=torso_target, right_target=right_target,
            left_target=left_target, head_target=head_target, duration=duration,
        )

    time.sleep(0.2)
    q_end = np.asarray(robot_init.get_state().position, dtype=np.float64)
    err_t = float(np.linalg.norm(q_end[TORSO_SLICE] - TORSO_INIT))
    err_r = float(np.linalg.norm(q_end[RIGHT_SLICE] - INIT_RIGHT))
    err_l = float(np.linalg.norm(q_end[LEFT_SLICE] - INIT_LEFT))
    print(f"[init-pose] done | final error norm torso={err_t:.6f}, right={err_r:.6f}, left={err_l:.6f}")

    if USE_REMOTE_GRIPPER:
        for _arm in ("right", "left"):
            ok_v = robot_init.set_tool_flange_output_voltage(_arm, 12)
            print(f"[init-pose] tool flange 12V {_arm}: {'OK' if ok_v else 'FAILED'}")
        time.sleep(0.5)

    try:
        robot_init.cancel_control()
    except Exception:
        pass
    if hasattr(robot_init, "disconnect"):
        robot_init.disconnect()

    INITIAL_POSE_DONE = True
else:
    INITIAL_POSE_DONE = False
    print("[init-pose] skipped")

# [Gripper Initialization] (episode 후 재초기화용)

In [ ]:
# ---------- Step 2-5: Gripper 초기화 (homing + 열기) ----------
# 전제: Step 2 (initial pose) 실행 완료 -> tool flange 12V 공급 완료
if not globals().get("INITIAL_POSE_DONE", False):
    raise RuntimeError("먼저 Step 2 (initial pose) 셀을 실행하세요.")
if not globals().get("USE_REMOTE_GRIPPER", False):
    print("[gripper-init] USE_REMOTE_GRIPPER=False -> skip")
else:
    from evaluation.rby1.remote_gripper import Gripper as RemoteGripper

    # Step 2에서 남아있는 stream 잔여 객체 제거
    # stream이 살아있으면 priority=10 control을 잡고 있어 이후 env 명령을 차단할 수 있음
    if "stream" in globals() and stream is not None:
        try:
            del stream
        except Exception:
            pass
        stream = None
        print("[gripper-init] 이전 stream 객체 정리 완료")

    # 이전 gripper 객체가 있으면 정리
    if "gripper_obj" in globals() and gripper_obj is not None:
        try:
            gripper_obj.stop()
            print("[gripper-init] 이전 gripper 객체 stop() 완료")
        except Exception as _e:
            print(f"[gripper-init] 이전 gripper stop 무시: {_e}")
        gripper_obj = None

    print("[gripper-init] RemoteGripper 연결 중...")
    gripper_obj = RemoteGripper()
    print(f"[gripper-init] host={gripper_obj.host}  port={gripper_obj.port}")

    if gripper_obj.host is None or gripper_obj.port is None:
        raise RuntimeError(
            "[gripper-init] gripper host/port가 설정되지 않았습니다.\n"
            "  scripts/deployment/config.yaml 또는 환경변수 확인"
        )

    # 1) ping / initialize
    print("[gripper-init] ping 확인 중...")
    ok_init = gripper_obj.initialize(verbose=True)
    print(f"[gripper-init] ping 결과: {'OK' if ok_init else 'FAILED'}")
    if not ok_init:
        raise RuntimeError("[gripper-init] 그리퍼 서버 응답 없음.")

    # 2) homing (범위 캘리브레이션)
    print("[gripper-init] homing 중... (30초 이내)")
    ok_homing = gripper_obj.homing()
    if not ok_homing:
        raise RuntimeError("[gripper-init] homing 실패")

    # homing 결과로 얻은 min_q / max_q 확인
    _min_q = getattr(gripper_obj, "min_q", None)
    _max_q = getattr(gripper_obj, "max_q", None)
    if _min_q is None or _max_q is None:
        raise RuntimeError(
            "[gripper-init] homing 성공했지만 min_q/max_q가 없음.\n"
            "  gripper_server.py 응답 형식에 min_q/max_q 필드가 있는지 확인하세요."
        )
    print(f"[gripper-init] homing 완료 | min_q={_min_q}  max_q={_max_q}")

    # 3) 제어 루프 시작
    print("[gripper-init] start() 호출 중...")
    gripper_obj.start()
    print("[gripper-init] start() 완료")

    # 4) 완전 열기/상태확인 구간 timeout 단축
    _old_timeout = getattr(gripper_obj, "timeout", None)
    _fast_timeout = 3.0
    try:
        _old_timeout_f = float(_old_timeout) if _old_timeout is not None else None
    except Exception:
        _old_timeout_f = None
    if _old_timeout_f is None or _old_timeout_f > _fast_timeout:
        gripper_obj.timeout = _fast_timeout
        try:
            gripper_obj._sock.settimeout(_fast_timeout)
        except Exception:
            pass
    print(f"[gripper-init] fast timeout 적용: {_old_timeout} -> {gripper_obj.timeout}")

    # 완전 열기 (normalized_q=1.0 -> OPEN) -- wait_for_reply=True로 확인
    print("[gripper-init] 완전 열기 명령 전송 (normalized=1.0)...")
    gripper_obj.set_normalized_target(np.array([1.0, 1.0]), wait_for_reply=True)
    time.sleep(0.5)

    # 현재 그리퍼 상태 확인
    try:
        _grip_state = gripper_obj.get_state()
        print(f"[gripper-init] 현재 그리퍼 상태: {_grip_state}  (열기 후 확인)")
    except Exception as _se:
        print(f"[gripper-init][WARN] 상태 조회 실패: {_se}")

    # 현재 normalized target 확인
    try:
        _grip_norm = gripper_obj.get_normalized_target()
        print(f"[gripper-init] 현재 normalized target: {_grip_norm}  (1.0=OPEN 이어야 함)")
    except Exception as _ne:
        print(f"[gripper-init][WARN] normalized target 조회 실패: {_ne}")

    GRIPPER_INIT_DONE = True
    print("[gripper-init] 완료")

---
## Action 로그 플롯 — 관절 궤적 및 Chunk 경계 분석

Episode loop 실행 후 `_log_*` 버퍼를 기반으로 3종류 그래프를 생성합니다.

| 그래프 | 내용 |
|--------|------|
| 전체 궤적 | 관절별 commanded vs actual (deg) |
| Step-to-step delta | chunk 경계 spike 확인 |
| Chunk 경계 zoom-in | 경계 전후 궤적 확대 |

In [ ]:
# ---------- Action chunk 로그 시각화 ----------
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

if "_log_cmd_right" not in globals() or len(_log_cmd_right) == 0:
    print("[plot] 로그 데이터 없음 — episode loop를 먼저 실행하세요.")
else:
    cmd_r    = np.array(_log_cmd_right)
    cmd_l    = np.array(_log_cmd_left)
    grip_r   = np.array(_log_cmd_grip_r)
    grip_l   = np.array(_log_cmd_grip_l)
    act_r    = np.array(_log_actual_right) if _log_actual_right else np.empty((0, 7))
    act_l    = np.array(_log_actual_left)  if _log_actual_left  else np.empty((0, 7))
    act_x    = np.array(_log_actual_step,  dtype=int)
    c_starts = np.array(_log_chunk_start,  dtype=int)

    N = cmd_r.shape[0]
    steps_x = np.arange(N)

    print(f"[plot] steps: {N}  |  chunks: {len(c_starts)}  |  actual samples: {len(act_x)}")

    def _plot_arm_full(title, cmd, act, act_steps, grip, tag):
        fig, axes = plt.subplots(2, 4, figsize=(20, 8), sharex=True)
        fig.suptitle(title, fontsize=13, fontweight="bold")
        axes_flat = axes.flatten()
        for j in range(7):
            ax = axes_flat[j]
            ax.plot(steps_x, np.degrees(cmd[:, j]), color="steelblue", lw=1.2, label="commanded")
            if len(act_steps) > 0:
                ax.scatter(act_steps, np.degrees(act[:, j]), color="tomato", s=18, zorder=5, label="actual")
            for cs in c_starts:
                ax.axvline(x=cs, color="dimgray", linestyle="--", lw=0.8, alpha=0.7)
            ax.set_title(f"{tag}_J{j}", fontsize=10)
            ax.set_ylabel("deg", fontsize=8)
            ax.tick_params(labelsize=7)
            ax.grid(True, alpha=0.3)
        ax_g = axes_flat[7]
        ax_g.plot(steps_x, grip, color="darkorange", lw=1.2)
        for cs in c_starts:
            ax_g.axvline(x=cs, color="dimgray", linestyle="--", lw=0.8, alpha=0.7)
        ax_g.set_title(f"{tag}_Gripper", fontsize=10)
        ax_g.set_ylabel("value", fontsize=8)
        ax_g.tick_params(labelsize=7)
        ax_g.grid(True, alpha=0.3)
        for ax in axes[1]:
            ax.set_xlabel("step", fontsize=8)
        handles = [
            mpatches.Patch(color="steelblue", label="commanded"),
            mpatches.Patch(color="tomato",    label="actual (sampled)"),
            plt.Line2D([0], [0], color="dimgray", linestyle="--", label="chunk start"),
        ]
        fig.legend(handles=handles, loc="lower center", ncol=3, fontsize=9, bbox_to_anchor=(0.5, -0.01))
        plt.tight_layout(rect=[0, 0.04, 1, 1])
        plt.show()

    _plot_arm_full("Right Arm (deg)", cmd_r, act_r, act_x, grip_r, "R")
    _plot_arm_full("Left Arm (deg)",  cmd_l, act_l, act_x, grip_l, "L")

    # Step-to-step delta norm
    delta_r = np.linalg.norm(np.diff(cmd_r, axis=0), axis=1)
    delta_l = np.linalg.norm(np.diff(cmd_l, axis=0), axis=1)

    fig2, (ax2r, ax2l) = plt.subplots(2, 1, figsize=(16, 6), sharex=True)
    fig2.suptitle("Step-to-step Command Delta (L2 norm, rad)", fontsize=11, fontweight="bold")
    for ax2, delta, color, arm_tag in [
        (ax2r, delta_r, "steelblue", "Right"),
        (ax2l, delta_l, "coral",     "Left"),
    ]:
        ax2.plot(np.arange(N - 1), delta, color=color, lw=0.9, label=f"{arm_tag} delta")
        for k, cs in enumerate(c_starts[1:]):
            ax2.axvline(x=cs - 1, color="red", linestyle="--", lw=1.5, alpha=0.85,
                        label="chunk boundary" if k == 0 else None)
        ax2.set_ylabel("delta norm (rad)", fontsize=9)
        ax2.set_title(f"{arm_tag} Arm", fontsize=10)
        ax2.grid(True, alpha=0.3)
        ax2.legend(fontsize=8)
    ax2l.set_xlabel("step", fontsize=9)
    plt.tight_layout()
    plt.show()

    print("\n[plot] === Step delta stats (chunk boundary vs internal) ===")
    for arm_tag, delta in [("Right", delta_r), ("Left", delta_l)]:
        in_mask = np.ones(len(delta), dtype=bool)
        for cs in c_starts[1:]:
            if 0 < cs - 1 < len(in_mask):
                in_mask[cs - 1] = False
        b_delta = delta[~in_mask]
        i_delta = delta[in_mask]
        print(
            f"  {arm_tag}: in-chunk avg={i_delta.mean():.4f} rad | "
            f"boundary avg={b_delta.mean() if len(b_delta) else 0:.4f} rad | "
            f"boundary max={b_delta.max()  if len(b_delta) else 0:.4f} rad"
        )

---
## Root Cause: Independent Chunk Re-sampling

**Hypothesis**: The spike is not a tracking or proprio issue — it is caused by the diffusion model independently sampling each chunk from Gaussian noise, with no continuity constraint between chunks.

```
chunk k   : [step 0 ──── step 15] [step 16 ─── step 99]  ← steps 16–99 are DISCARDED
chunk k+1 : [step 0 ──── step 15] ...                      ← freshly sampled from noise
               ↑
         spike here: chunk_k[15] vs chunk_k+1[0]
         = two completely independent denoising trajectories
```

**Key test**: Compare the gap at the actual boundary (`chunk_k[exec_size-1] → chunk_k+1[0]`)  
against the gap that *would have been seamless* within the same prediction (`chunk_k[exec_size-1] → chunk_k[exec_size]`).

- If **within-chunk gap << cross-chunk gap**: the model *is* consistent internally — re-sampling is the culprit
- The ratio of these two gaps quantifies how much "continuity" is lost by discarding the un-executed tail


In [ ]:
# ---------- Independent re-sampling hypothesis: within-chunk vs cross-chunk continuity ----------
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

if "_log_full_chunks" not in globals() or len(_log_full_chunks) < 2:
    print("[resample] _log_full_chunks not available — re-run the episode loop first.")
else:
    c_starts  = np.array(_log_chunk_start, dtype=int)
    n_chunks  = len(_log_full_chunks)
    exec_size = EXECUTE_CHUNK_SIZE

    # ── Per-boundary: within-chunk gap vs cross-chunk gap ──────────────
    # within:  chunk_k[exec_size-1] → chunk_k[exec_size]   (same denoising, step boundary inside)
    # cross:   chunk_k[exec_size-1] → chunk_k+1[0]         (independent re-sample)

    within_r, cross_r = [], []
    within_l, cross_l = [], []
    bnd_labels = []

    for k in range(n_chunks - 1):
        fc_k   = _log_full_chunks[k]
        fc_k1  = _log_full_chunks[k + 1]

        T = fc_k["right"].shape[0]
        last_exec = exec_size - 1
        next_step = exec_size  # what chunk_k[exec_size] would have been

        if next_step >= T:
            continue  # chunk shorter than execute_size, skip

        # within-chunk: one step further inside the same prediction
        within_r.append(float(np.linalg.norm(fc_k["right"][next_step] - fc_k["right"][last_exec])))
        within_l.append(float(np.linalg.norm(fc_k["left"][next_step]  - fc_k["left"][last_exec])))

        # cross-chunk: first step of next independent prediction
        cross_r.append(float(np.linalg.norm(fc_k1["right"][0] - fc_k["right"][last_exec])))
        cross_l.append(float(np.linalg.norm(fc_k1["left"][0]  - fc_k["left"][last_exec])))

        bnd_labels.append(f"B{k+1}")

    if len(within_r) == 0:
        print("[resample] All chunks shorter than EXECUTE_CHUNK_SIZE — cannot compare.")
    else:
        bnd_x  = np.arange(len(within_r))
        w      = 0.35

        fig, axes = plt.subplots(2, 1, figsize=(max(10, len(within_r) * 0.9), 8), sharex=True)
        fig.suptitle(
            f"Within-chunk vs Cross-chunk Step Gap at Boundary  (exec_size={exec_size})\n"
            f"within: chunk_k[{exec_size-1}]→chunk_k[{exec_size}]   "
            f"cross: chunk_k[{exec_size-1}]→chunk_k+1[0]",
            fontsize=11, fontweight="bold"
        )

        for ax, tag, wi, cr in [(axes[0], "Right", within_r, cross_r),
                                 (axes[1], "Left",  within_l, cross_l)]:
            ax.bar(bnd_x - w/2, wi, w, label=f"within-chunk (step {exec_size-1}→{exec_size})", color="steelblue", alpha=0.85)
            ax.bar(bnd_x + w/2, cr, w, label=f"cross-chunk  (step {exec_size-1} → next chunk[0])", color="tomato", alpha=0.85)

            for i, (wi_v, cr_v) in enumerate(zip(wi, cr)):
                ratio = cr_v / max(wi_v, 1e-8)
                ax.text(i + w/2, cr_v + 0.001, f"{ratio:.1f}x",
                        ha="center", va="bottom", fontsize=8, color="tomato", fontweight="bold")

            ax.set_ylabel("L2 norm (rad)")
            ax.set_title(f"{tag} Arm   —   avg within={np.mean(wi):.4f}  avg cross={np.mean(cr):.4f}  "
                         f"ratio={np.mean(cr)/max(np.mean(wi), 1e-8):.1f}x")
            ax.legend(fontsize=8)
            ax.grid(axis="y", alpha=0.3)

        axes[1].set_xticks(bnd_x)
        axes[1].set_xticklabels(bnd_labels, fontsize=8)
        axes[1].set_xlabel("Boundary index")
        plt.tight_layout()
        plt.show()

        # ── Visualize: full chunk_k (no split) + chunk_k+1 from discard point ──
        SHOW_BNDIDX = 0   # which boundary to zoom in on
        LOOK_AFTER  = min(exec_size, _log_full_chunks[SHOW_BNDIDX + 1]["right"].shape[0])

        fc_k  = _log_full_chunks[SHOW_BNDIDX]
        fc_k1 = _log_full_chunks[SHOW_BNDIDX + 1]
        T_k   = fc_k["right"].shape[0]

        # x=0 = discard point (= exec_size step of chunk k)
        # chunk k full prediction: x from -exec_size to T_k - exec_size - 1 (one continuous line)
        # chunk k+1: x from 0 to LOOK_AFTER - 1
        x_chunk_k  = np.arange(T_k) - exec_size      # [-exec_size, ..., T_k-exec_size-1]
        x_chunk_k1 = np.arange(LOOK_AFTER)            # [0, ..., LOOK_AFTER-1]

        fig2, axes2 = plt.subplots(2, 7, figsize=(26, 7), sharex=False)
        fig2.suptitle(
            f"Boundary B{SHOW_BNDIDX+1} zoom-in  |  blue = chunk k full prediction  |  red = chunk k+1\n"
            f"x=0 = discard point   |   shaded region (x<0) = executed portion of chunk k",
            fontsize=11, fontweight="bold"
        )

        for arm_idx, (arm_tag, key) in enumerate([("Right", "right"), ("Left", "left")]):
            for j in range(7):
                ax = axes2[arm_idx, j]

                seg_full_k = np.degrees(fc_k[key][:, j])            # entire chunk k
                seg_next   = np.degrees(fc_k1[key][:LOOK_AFTER, j]) # chunk k+1

                ax.plot(x_chunk_k,  seg_full_k, color="steelblue", lw=1.5, label="chunk k")
                ax.plot(x_chunk_k1, seg_next,   color="tomato",    lw=1.5, label="chunk k+1")
                ax.axvline(0, color="black", lw=1.0, linestyle=":")
                ax.axvspan(-exec_size, 0, alpha=0.08, color="steelblue")  # executed region shading

                ax.set_title(f"{arm_tag} J{j}", fontsize=9)
                ax.set_ylabel("deg", fontsize=7)
                ax.tick_params(labelsize=7)
                ax.grid(True, alpha=0.3)

        handles = [
            mpatches.Patch(facecolor="steelblue", alpha=0.3, label=f"chunk k executed region (x<0)"),
            plt.Line2D([0], [0], color="steelblue", lw=2, label="chunk k full prediction"),
            plt.Line2D([0], [0], color="tomato",    lw=2, label="chunk k+1"),
            plt.Line2D([0], [0], color="black",     lw=1, linestyle=":", label="discard point (x=0)"),
        ]
        fig2.legend(handles=handles, loc="lower center", ncol=4, fontsize=9, bbox_to_anchor=(0.5, -0.02))
        plt.tight_layout(rect=[0, 0.06, 1, 1])
        plt.show()

        print(f"[resample] zoom-in: T_k={T_k}, exec_size={exec_size}, LOOK_AFTER={LOOK_AFTER}")

        # ── Text summary ────────────────────────────────────────────────
        print("\n[resample] ====== Within vs Cross gap summary ======")
        for tag, wi, cr in [("Right", within_r, cross_r), ("Left", within_l, cross_l)]:
            wi_arr, cr_arr = np.array(wi), np.array(cr)
            ratio = cr_arr.mean() / max(wi_arr.mean(), 1e-8)
            print(f"  [{tag}]  within avg={wi_arr.mean():.4f}  cross avg={cr_arr.mean():.4f}  ratio={ratio:.1f}x")
            if ratio > 3.0:
                verdict = "CONFIRMED: independent re-sampling is the dominant cause of boundary spikes."
            elif ratio > 1.5:
                verdict = "LIKELY: re-sampling contributes significantly, but other factors also present."
            else:
                verdict = "UNCLEAR: within/cross gap similar — spike may come from a different source."
            print(f"  -> {verdict}")


## Cell 10 — 정리 (Cleanup)

In [ ]:
# Stop gripper
if "gripper_obj" in globals() and gripper_obj is not None:
    try:
        gripper_obj.stop()
        print("[cleanup] gripper stopped")
    except Exception as e:
        print(f"[cleanup] gripper stop failed: {e}")
    gripper_obj = None

# Stop cameras
if "cam_pipes" in globals() and cam_pipes:
    _stop_cameras(cam_pipes)
    cam_pipes = {}
    print("[cleanup] cameras stopped")

# Disconnect robot
if "robot" in globals() and robot is not None:
    try:
        robot.cancel_control()
    except Exception:
        pass
    if hasattr(robot, "disconnect"):
        robot.disconnect()
    robot = None
    print("[cleanup] robot disconnected")

# Reset flags
INITIAL_POSE_DONE = False
GRIPPER_INIT_DONE = False
CAM_SETUP_DONE    = False
ENV_SETUP_DONE    = False

print("[cleanup] done")